In [0]:
from pyspark.sql import functions as F

### widgets --> to parametrize databricks notebook

In [0]:
dbutils.widgets.text("catalog_name","ecommerce","Catalog Name")
dbutils.widgets.text("storage_account_name","eccommerce_storage","Storage Account Name")
dbutils.widgets.text("container_name","eccom-raw-data","Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

print(catalog_name, storage_account_name, container_name)

ecommerce stgecomadlsddevap001 ecomm-raw-data


In [0]:
# Azure datalake storage
adls_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/order_returns/landing/"

# checkpoint folders for streaming (bronze,silver,gold)
bronze_checkpoint_path =  f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/bronze/fact_returns/"


In [0]:
spark.readStream \
 .format("cloudFiles") \
 .option("cloudFiles.format", "csv")  \
 .option("cloudFiles.schemaLocation", bronze_checkpoint_path) \
 .option("cloudFiles.schemaEvolutionMode", "rescue") \
 .option("header", "true") \
 .option("cloudFiles.inferColumnTypes", "true") \
 .option("rescuedDataColumn", "_rescued_data") \
 .option("cloudFiles.includeExistingFiles", "true")  \
 .option("pathGlobFilter", "*.csv") \
 .load(adls_path) \
 .withColumn("ingest_timestamp", F.current_timestamp()) \
 .withColumn("source_file", F.col("_metadata.file_path")) \
 .writeStream \
 .outputMode("append") \
 .option("checkpointLocation", bronze_checkpoint_path) \
 .trigger(availableNow=True) \
 .toTable(f"{catalog_name}.bronze.brz_returns") \
 .awaitTermination()